# Poglavje 7: Gradnja klepetalnih aplikacij
## Hitri začetek z OpenAI API

Ta zvezek je prilagojen iz [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vključuje zvezke za dostop do storitev [Azure OpenAI](notebook-azure-openai.ipynb).

Python OpenAI API deluje tudi z Azure OpenAI modeli, z nekaj prilagoditvami. Več o razlikah si preberite tukaj: [Kako preklapljati med OpenAI in Azure OpenAI končnimi točkami s Pythonom](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodni niz besedila velik jezikovni model poskuša napovedati besedilo, ki bo sledilo" (1). Ta "hitri zagon" zvezek bo uporabnike uvedel v osnovne koncepte LLM, osnovne zahteve paketov za začetek z AML, mehki uvod v oblikovanje pozivov in več kratkih primerov različnih primerov uporabe. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati storitev OpenAI](#how-to-use-openai-service)  
[1. Ustvarjanje vaše storitve OpenAI](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Overitve](#3.-credentials)  

[Primeri uporabe](#use-cases)    
[1. Povzemanje besedila](#1.-summarize-text)  
[2. Razvrščanje besedila](#2.-classify-text)  
[3. Generiranje novih imen izdelkov](#3.-generate-new-product-names)  
[4. Izboljšava razvrščevalnika](#4.fine-tune-a-classifier)  

[Reference](#references)


### Ustvarite svoj prvi poziv  
Ta kratka vaja bo zagotovila osnovni uvod za pošiljanje pozivov modelu OpenAI za preprosto nalogo "povzemanja".


**Koraki**:  
1. Namestite knjižnico OpenAI v svoje Python okolje  
2. Naložite standardne pomožne knjižnice in nastavite svoje običajne varnostne poverilnice OpenAI za storitev OpenAI, ki ste jo ustvarili  
3. Izberite model za svojo nalogo  
4. Ustvarite preprost poziv za model  
5. Pošljite svojo zahtevo API-ju modela!


### 1. Namestite OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Uvozi pomožne knjižnice in ustvari poverilnice


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Iskanje pravega modela  
Modeli, kot sta GPT-4o in GPT-4o mini, lahko razumejo in ustvarjajo naravni jezik ter so na voljo na platformi OpenAI z različnimi stopnjami zmogljivosti in hitrosti, primernimi za različne naloge.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Oblikovanje pozivov  

"Čar velikih jezikovnih modelov je v tem, da se s treniranjem na minimiziranje napake v napovedovanju preko ogromnih količin besedil modeli naučijo konceptov, ki so uporabni za te napovedi. Na primer, naučijo se konceptov, kot so"(1):

* kako črkovati
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v več jezikih
* kako programirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Iz vseh vhodov v velik jezikovni model je daleč najbolj vpliven besedilni poziv(1).

Velike jezikovne modele lahko spodbudimo k izhodu na nekaj načinov:

Navodilo: Povej modelu, kaj želiš
Dokončanje: Zaključi začetek tistega, kar želiš
Demonstracija: Pokaži modelu, kaj želiš, z:
Nekaj primeri v pozivu
Stotine ali tisoče primerov v učni podatkovni zbirki za fino uglaševanje"



#### Obstajajo tri osnovna pravila za ustvarjanje pozivov:

**Pokaži in povej**. Jasno povej, kaj želiš, bodisi z navodili, primeri ali kombinacijo obojega. Če želiš, da model uredi seznam elementov po abecedi ali razvrsti odstavek po sentimentu, mu jasno pokaži, da to želiš.

**Posreduj kakovostne podatke**. Če želiš zgraditi klasifikator ali prinesti model, da sledi vzorcu, poskrbi, da je dovolj primerov. Preveri svoje primere — model je običajno dovolj pameten, da vidi osnovne pravopisne napake in ti da odgovor, vendar lahko tudi predpostavi, da je to namerno, kar lahko vpliva na odgovor.

**Preveri svoje nastavitve.** Nastavitve temperature in top_p nadzorujejo, kako determinističen je model pri generiranju odgovora. Če želiš odgovor, kjer obstaja samo en pravilen odgovor, želiš nastaviti nižje vrednosti. Če iščeš bolj raznolike odgovore, jih lahko nastaviš višje. Največja napaka ljudi pri teh nastavitvah je, da mislijo, da nadzorujejo "pametnost" ali "kreativnost".


Vir: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Oddaj!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Povzemanje besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odlomka dodate 'tl;dr:'. Opazite, kako model razume, kako opraviti različne naloge brez dodatnih navodil. Lahko eksperimentirate z bolj opisnimi ukazi kot je tl;dr, da spremenite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Nedavne raziskave so pokazale znatne izboljšave na mnogih NLP nalogah in merilih z vnaprejšnjim učenjem na velikem korpusu besedil, ki ji sledi fino nastavljanje na specifični nalogi. Čeprav je arhitektura običajno neodvisna od naloge, ta metoda vseeno zahteva zbirke podatkov za fino nastavljanje specifične za nalogo, ki obsegajo tisoče ali desetine tisoč primerov. Nasprotno pa ljudje običajno lahko opravijo novo jezikovno nalogo s samo nekaj primeri ali preprostimi navodili – nekaj, s čimer se trenutni NLP sistemi še vedno večinoma trudijo. Tukaj pokažemo, da velikost jezikovnih modelov znatno izboljša splošno, malo-učeno uspešnost, včasih celo konkurira z najboljšo prejšnjo metodo finotnega nastavljanja.  



Povzeto  


# Vaje za več primerov uporabe  
1. Povzetek besedila  
2. Razvrščanje besedila  
3. Generiranje novih imen izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Razvrsti besedilo  
#### Izziv  
Razvrsti predmete v kategorije, ki so podane v času sklepanja. V naslednjem primeru podamo tako kategorije kot tudi besedilo za razvrstitev v pozivu(*playground_reference). 

Povpraševanje stranke: Pozdravljeni, nedavno se je zlomila ena tipka na tipkovnici mojega prenosnika in potreboval bom nadomestno:

Razvrščena kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Ustvarite nove imena izdelkov
#### Izziv
Ustvarite imena izdelkov iz primerov besed. Tukaj vključimo v poziv informacije o izdelku, za katerega bomo generirali imena. Prav tako priložimo podoben primer, da pokažemo vzorec, ki ga želimo dobiti. Nastavili smo tudi visoko vrednost temperature, da povečamo naključnost in bolj inovativne odgovore.

Opis izdelka: Naprava za pripravo mlečnih napitkov doma
Začetne besede: hiter, zdrav, kompakten.
Imena izdelkov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: Par čevljev, ki se prilagodi katerikoli velikosti stopala.
Začetne besede: prilagodljiv, primeren, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino nastavljanje modelov GPT za klasifikacijo besedila](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Sodelujoči
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
